# Track 04 - ARC-AGI-2 Neuro-Symbolic Solver

Offline reproduction notebook for the **Dihedral-Invariant DSL Synthesis + Test-Time Adaptation** engine.

Pipeline per task:

1. D4 + colour-permutation canonicalisation (up to 16x search-space reduction).
2. Depth-1 DSL enumeration, bidirectional forward/backward state intersection,
   depth-2 beam enumeration (pair 0 is an O(1) exactness filter).
3. Block-mosaic fitting for `k x m` tiled outputs.
4. Test-time learned rules (per-cell decision tree, per-object decision tree,
   small MLP) - accepted only with a **perfect** training fit.
5. Majority-colour fallback for the second attempt.

Everything runs locally; no network access and no Kaggle submission.


In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "src" / "search.py").is_file():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))
print("track root:", ROOT)

import numpy, sklearn, torch
from src import dsl_primitives as dsl
from src.search import DslSolver, SolverConfig, load_tasks
print("numpy", numpy.__version__, "| sklearn", sklearn.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())


In [ ]:
DATA = ROOT / "data" / "arc-agi-2" / "data"
train_files = sorted((DATA / "training").glob("*.json"))
eval_files = sorted((DATA / "evaluation").glob("*.json"))
print(f"training tasks: {len(train_files)} | evaluation tasks: {len(eval_files)}")

sample = json.loads(train_files[0].read_text())
print("example task keys:", list(sample), "| train pairs:", len(sample["train"]))
print("input:\n", np.asarray(sample["train"][0]["input"]))
print("output:\n", np.asarray(sample["train"][0]["output"]))


## 1. Canonicalisation sanity check
Min over 8 D4 images plus colour relabelling must give one key per orbit.

In [ ]:
from src.canonical import D4, canonical_key
base = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.int8)
keys = {canonical_key(fn(base)) for _, fn in D4}
print("distinct canonical keys across D4 orbit:", len(keys))
assert len(keys) == 1


## 2. Self-check on synthetic trivial tasks
Identity, rotation, mirror, colour swap, upscale, tiling, gravity, crop, symmetry completion.

In [ ]:
import src.selfcheck as selfcheck
rc = selfcheck.main()
assert rc == 0


## 3. Solve one real task end-to-end

In [ ]:
task = load_tasks(DATA, split="training", limit=3)[2]
solver = DslSolver(SolverConfig(time_budget=10.0))
start = time.time()
result = solver.solve(task)
print(f"task {task.task_id} solved={result.solved} method={result.method} "
      f"runtime={result.runtime_s:.2f}s notes={result.notes!r}")
print("predicted attempt_1:\n", result.attempts[0][0])
print("fallback shape:", result.attempts[0][1].shape)


## 4. Reproduce / inspect the full run metrics
Runs a 12-task slice of the evaluation split if the stored metrics file is absent.

In [ ]:
from src.dsl_search_runner import run_split

metrics_path = ROOT / "outputs" / "eval_metrics.json"
if not metrics_path.is_file():
    config = SolverConfig(time_budget=8.0)
    run_split(str(DATA), "evaluation", str(ROOT / "outputs"), config, workers=4, max_wall=600)
metrics = json.loads(metrics_path.read_text())
summary = {k: metrics[k] for k in ("split", "tasks", "solved", "solve_rate", "wall_seconds",
                                   "graded_pass1_rate", "graded_pass2_rate", "by_method")}
print(json.dumps(summary, indent=2))


In [ ]:
import pandas as pd
rows = [{"signature": sig, "tasks": v["tasks"], "solved": v["solved"], "rate": v["rate"]}
        for sig, v in metrics["by_signature"].items()]
df = pd.DataFrame(rows).sort_values(["rate", "tasks"], ascending=False)
print(df.to_string(index=False))
print("\ntotal solved:", int(df.solved.sum()), "/", int(df.tasks.sum()))


## 5. Inspect the submission contract
`attempt_1` / `attempt_2` per evaluation task id (grid for a single test input, list of grids when a task has several).

In [ ]:
sub_path = ROOT / "outputs" / "submission.json"
if sub_path.is_file():
    submission = json.loads(sub_path.read_text())
    print("tasks in submission:", len(submission))
    first_id = sorted(submission)[0]
    print(first_id, "->", {k: (np.asarray(v).shape if isinstance(v, list) else type(v)) for k, v in submission[first_id].items()})
    covered = set(submission) == {p.stem for p in eval_files}
    print("every evaluation id covered:", covered)
else:
    print("submission.json not generated yet - run src.dsl_search_runner")


## 6. Notes

* Training split is the development signal; the evaluation split is only scored
  after the run (gold outputs are never visible to the solver).
* The solution is a cascade: symbolic hypotheses outrank learned ones, and both
  must fit every training pair exactly.
* Reported numbers come from `outputs/eval_metrics.json` (real run artefacts).
